In [ ]:
import os
import psycopg2
import pandas as pd
import plotly.express as px
import numpy as np
from dotenv import load_dotenv
from sklearn.impute import SimpleImputer

#Chargement des identifiants depuis le fichier d'environnement  (bloqué sur github par .gitignore de *.env)
load_dotenv("credential.env")
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT")
)

# récupération des données RDS kayak_data
df = pd.read_sql("SELECT * FROM kayak_data;", conn)
conn.close()

# gestion des valeurs manquantes
# remplacement par la moyenne 
imputer = SimpleImputer(strategy="mean")

# la colonne 'note' à des valeurs nan, remplacement par la note -> 8.5
df['note'] = imputer.fit_transform(df[['note']]).round(1)



### CARTE 1 : TOP 5 DESTINATIONS (en fonction du beau_temps mini)

In [ ]:

# regroupement par ville et données gps(pour plotly)
    #agrégation(par ville ) de la première valeur de beau_temps et de la moyenne des températures 
df_top5 = (
    df.groupby(['ville', 'latitude', 'longitude']) 
      .agg({'beau_temps': 'first', 'temperature': 'mean'}) 
      .reset_index() 
      .sort_values('beau_temps') 
      .head(5) 
)


# gros problème de visibilité car les villes se chevauchent sur la carte (Marseille / Cassis / Bormes-les-Mimosas)
#la solution consiste à arbitrairement décaler leur position gps sur la carte (ici de l'aléatoire + ou - 0.700 ce qui est beaucoup)
df_top5['latitude'] = df_top5['latitude'] + np.random.uniform(-0.700, 0.700, len(df_top5)) 
df_top5['longitude'] = df_top5['longitude'] + np.random.uniform(-0.700, 0.700, len(df_top5))

# création de la carte
fig_villes = px.scatter_mapbox(
    df_top5,
    lat="latitude",
    lon="longitude",
    size="beau_temps",
    color="temperature",
    color_continuous_scale=["blue", "purple", "red"],
    opacity=0.7, size_max=15, zoom=4,
    mapbox_style="carto-positron",
    text='ville',
    title="TOP 5 DESTINATIONS (en fonction du beau_temps)"
)


# amélioration du cadrage sur le centre absolu entre les villes
fig_villes.update_layout(
    margin={"r":0,"t":40,"l":0,"b":0}, # refonte des marges extérieurs de la carte pour améliorer le coté 'fullscreen'
    mapbox=dict(
        center=dict(
            lat=df_top5['latitude'].mean(), #le centre de la carte est la moyenne des coordonnées gps
            lon=df_top5['longitude'].mean()
        )
    )
)


fig_villes.show()



C:\Users\julie\AppData\Local\Temp\ipykernel_2988\708386002.py:12: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



### CARTE 2 : TOP 20 HOTELS 

In [249]:
# création d'un mask des tops 5 villes de destination
top_villes = df_top5['ville'].tolist()

# top 20 des hôtels parmis les 5 villes
df_top20_hotels = df[df['ville'].isin(top_villes)].sort_values(by='note', ascending=False).head(20).copy()


# gros problème de visibilité car les hôtels se chevauche sur la carte
#la solution consiste à arbitrairement décaler leur position gps sur la carte 
df_top20_hotels['latitude'] = df_top20_hotels['latitude'] + np.random.uniform(-0.150, 0.150, len(df_top20_hotels)) 
df_top20_hotels['longitude'] = df_top20_hotels['longitude'] + np.random.uniform(-0.150, 0.150, len(df_top20_hotels))

# pour se répérer plus facillement j'ai ajouté la ville après le nom de l'hôtel lors du survol avec la sourie
df_top20_hotels['titre_au_survol'] = df_top20_hotels['nom_hotel'] + " (" + df_top20_hotels['ville'] + ")" 


# Création de la carte des hôtels
fig_hotels = px.scatter_mapbox(
    df_top20_hotels,
    lat="latitude",
    lon="longitude",
    hover_name="titre_au_survol",
    size="note",
    color="note",
    color_continuous_scale=["blue", "purple", "red"],
    opacity=0.7,
    size_max=20, 
    zoom=4.7,
    mapbox_style="carto-positron",
    title="Top 20 des meilleurs hôtels"
)

# amélioration du cadrage sur les hôtels
fig_hotels.update_layout(
    margin={"r":0,"t":40,"l":0,"b":0}, # refonte des marges extérieurs de la carte pour améliorer le coté 'fullscreen'
    mapbox=dict(
        center=dict(
            lat=df_top20_hotels['latitude'].mean(), #le centre de la carte est la moyenne des coordonnées gps
            lon=df_top20_hotels['longitude'].mean()
        )
    )
)

fig_hotels.show()

C:\Users\julie\AppData\Local\Temp\ipykernel_2988\1793669270.py:18: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/

